# NeuroGolf 2026 — Competitive Solver v2

Strategy to beat 4630 pts:
1. **Handcrafted ONNX graphs** for detected geometric/color patterns (zero training cost)
2. **Least-squares linear** on all 260+ examples for small grids
3. **Trained single-layer conv** (1×1→7×7) on ALL data including arc-gen
4. **Multi-layer CNNs** with residual connections as fallback
5. Always validate against ALL examples before accepting
6. Always prefer cheaper solution (lower MACs+params+memory)

In [1]:
import os, json, math, copy, time, zipfile, warnings
import numpy as np
from pathlib import Path
import onnx, onnxruntime
import onnx.helper as oh
import onnx.numpy_helper as onh
from onnx import TensorProto
import torch, torch.nn as nn
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────
TASK_DIR   = Path('/Users/manishswami/developer/Nuero-golf championship/neurogolf-2026')
OUTPUT_DIR = Path('./submission')
SUB_DIR    = OUTPUT_DIR / 'submission'
SUB_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────
C, H, W    = 10, 30, 30
OPSET      = oh.make_opsetid('', 17)
IR_VER     = 8
MAX_BYTES  = int(1.44 * 1024 * 1024)  # 1.44 MB

DEVICE = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('mps') if torch.backends.mps.is_available()
          else torch.device('cpu'))
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: mps


In [2]:
# ── Data helpers ───────────────────────────────────────────────────

def load_task(path):
    with open(path) as f:
        return json.load(f)

def g2t(grid):  # grid → (C,H,W) float32 numpy
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((C, H, W), dtype=np.float32)
    for r in range(h):
        for c in range(w):
            if 0 <= g[r,c] <= 9:
                t[g[r,c], r, c] = 1.0
    return t

def g2t_batch(pairs, key):  # list of pairs → (N,C,H,W)
    return np.stack([g2t(p[key]) for p in pairs])

def t2grid(t, oh, ow):  # (1,C,H,W) or (C,H,W) → grid
    arr = t[0] if t.ndim == 4 else t
    grid = []
    for r in range(oh):
        row = []
        for c in range(ow):
            col = arr[:, r, c]
            row.append(int(np.argmax(col)) if col.max() >= 0.5 else 0)
        grid.append(row)
    return grid

# ── ONNX helpers ───────────────────────────────────────────────────

def make_model(nodes, inits=None):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1,C,H,W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1,C,H,W])
    graph = oh.make_graph(nodes, 'g', [X], [Y], initializer=inits or [])
    model = oh.make_model(graph, opset_imports=[OPSET])
    model.ir_version = IR_VER
    return model

def save_onnx(model, path):
    path = Path(path)
    with open(path, 'wb') as f:
        f.write(model.SerializeToString())
    return path.stat().st_size

def run_onnx(model_bytes_or_path, pairs):
    """Run ONNX model against all pairs. Returns (n_correct, n_total)."""
    try:
        if isinstance(model_bytes_or_path, (str, Path)):
            sess = onnxruntime.InferenceSession(str(model_bytes_or_path),
                                                providers=['CPUExecutionProvider'])
        else:
            buf = model_bytes_or_path.SerializeToString()
            sess = onnxruntime.InferenceSession(buf, providers=['CPUExecutionProvider'])
    except Exception as e:
        return 0, len(pairs)
    correct = 0
    for p in pairs:
        inp = g2t(p['input'])[np.newaxis]  # (1,C,H,W)
        og  = np.array(p['output'])
        oh_, ow_ = og.shape
        try:
            pred = sess.run(None, {'input': inp})[0]
            pred_grid = t2grid(pred, oh_, ow_)
            if pred_grid == p['output']:
                correct += 1
        except Exception:
            pass
    return correct, len(pairs)

def check_all(model, all_pairs):
    c, t = run_onnx(model, all_pairs)
    return c == t

def score_model(path):
    try:
        import onnx_tool
        m = onnx_tool.loadmodel(str(path), {'verbose': False})
        g = m.graph
        g.graph_reorder_nodes(); g.shape_infer(None); g.profile()
        cost = int(sum(g.macs)) + int(g.memory) + int(g.params)
        return max(1.0, 25.0 - math.log(max(cost, 1)))
    except Exception:
        return 1.0

print('Helpers loaded.')

Helpers loaded.


In [ ]:
# ── Gather-based pixel permutation ─────────────────────────────────
# Core ONNX primitive: permute pixels with a Gather on flattened CHW.
# gather_indices[dst] = src  means output pixel dst comes from input pixel src.

def _gather_model(indices):  # indices: (H*W,) int64
    sh_chw  = onh.from_array(np.array([1,C,H*W], np.int64), name='sh_chw')
    sh_out  = onh.from_array(np.array([1,C,H,W], np.int64), name='sh_out')
    gi      = onh.from_array(indices.astype(np.int64),        name='gi')
    nodes = [
        oh.make_node('Constant', [], ['sh_chw'], value=sh_chw),
        oh.make_node('Reshape',  ['input','sh_chw'], ['flat']),
        oh.make_node('Constant', [], ['gi'], value=gi),
        oh.make_node('Gather',   ['flat','gi'], ['gathered'], axis=2),
        oh.make_node('Constant', [], ['sh_out'], value=sh_out),
        oh.make_node('Reshape',  ['gathered','sh_out'], ['output']),
    ]
    return make_model(nodes)

def _perm_from_fn(fn):  # fn(r,c) -> (new_r, new_c)
    idx = np.zeros(H*W, np.int64)
    for r in range(H):
        for c in range(W):
            nr, nc = fn(r, c)
            if 0 <= nr < H and 0 <= nc < W:
                idx[nr*W + nc] = r*W + c
    return idx

# ── Handcrafted models ─────────────────────────────────────────────

def model_identity():
    return make_model([oh.make_node('Identity', ['input'], ['output'])])

def model_color_remap(cmap):  # {src_color: dst_color}
    W = np.zeros((C,C,1,1), np.float32)
    for s, d in cmap.items():
        W[d,s,0,0] = 1.0
    for ch in range(C):
        if ch not in cmap:
            W[ch,ch,0,0] = 1.0
    wt = onh.from_array(W, name='W')
    nodes = [
        oh.make_node('Constant', [], ['W'], value=wt),
        oh.make_node('Conv', ['input','W'], ['output'],
                     kernel_shape=[1,1], pads=[0,0,0,0]),
    ]
    return make_model(nodes)

def model_rot90():   return _gather_model(_perm_from_fn(lambda r,c: (c, H-1-r)))
def model_rot180():  return _gather_model(_perm_from_fn(lambda r,c: (H-1-r, W-1-c)))
def model_rot270():  return _gather_model(_perm_from_fn(lambda r,c: (W-1-c, r)))
def model_hflip():   return _gather_model(_perm_from_fn(lambda r,c: (r, W-1-c)))
def model_vflip():   return _gather_model(_perm_from_fn(lambda r,c: (H-1-r, c)))
def model_hvflip():  return _gather_model(_perm_from_fn(lambda r,c: (H-1-r, W-1-c)))
def model_transpose():return _gather_model(_perm_from_fn(lambda r,c: (c, r)))
def model_antitranspose(): return _gather_model(_perm_from_fn(lambda r,c: (W-1-c, H-1-r)))

def model_crop(r0, c0, oh_, ow_):
    idx = np.zeros(H*W, np.int64)
    for r in range(H):
        for c in range(W):
            if r < oh_ and c < ow_:
                sr, sc = r0+r, c0+c
                idx[r*W+c] = sr*W+sc if sr<H and sc<W else 0
    # zero out area outside crop
    mask = np.zeros((1,C,H,W), np.float32)
    mask[0,:,:oh_,:ow_] = 1.0
    gi = onh.from_array(idx, name='gi')
    sh_chw = onh.from_array(np.array([1,C,H*W], np.int64), name='sh_chw')
    sh_out = onh.from_array(np.array([1,C,H,W], np.int64), name='sh_out')
    mk = onh.from_array(mask, name='mask')
    nodes = [
        oh.make_node('Constant',[], ['sh_chw'], value=sh_chw),
        oh.make_node('Reshape', ['input','sh_chw'], ['flat']),
        oh.make_node('Constant',[], ['gi'], value=gi),
        oh.make_node('Gather',  ['flat','gi'], ['gath'], axis=2),
        oh.make_node('Constant',[], ['sh_out'], value=sh_out),
        oh.make_node('Reshape', ['gath','sh_out'], ['raw']),
        oh.make_node('Constant',[], ['mask'], value=mk),
        oh.make_node('Mul', ['raw','mask'], ['output']),
    ]
    return make_model(nodes)

def model_scale(factor):  # integer scale-up by pixel repetition
    idx = np.zeros(H*W, np.int64)
    for r in range(H):
        for c in range(W):
            sr, sc = r//factor, c//factor
            idx[r*W+c] = sr*W+sc
    return _gather_model(idx)

def model_tile(tr, tc):   # repeat input tr×tc times
    idx = np.zeros(H*W, np.int64)
    for r in range(H):
        for c in range(W):
            idx[r*W+c] = (r % (H//tr)) * W + (c % (W//tc))
    return _gather_model(idx)

def model_const(output_tensor):  # constant output regardless of input
    # output = 0*input + const  — makes input 'used' so ONNX is valid
    ct = onh.from_array(output_tensor.astype(np.float32), name='ct')
    zr = onh.from_array(np.zeros((1,C,H,W), np.float32), name='zr')
    nodes = [
        oh.make_node('Constant',[], ['ct'], value=ct),
        oh.make_node('Constant',[], ['zr'], value=zr),
        oh.make_node('Mul', ['input','zr'], ['zeroed']),
        oh.make_node('Add', ['zeroed','ct'], ['output']),
    ]
    return make_model(nodes)

def model_gather_custom(indices):  # raw gather from detected permutation
    return _gather_model(indices)

print('Handcrafted models ready.')

In [ ]:
# ── Pattern detectors ──────────────────────────────────────────────

def _all_same_size(pairs):
    s = set()
    for p in pairs:
        s.add((np.array(p['input']).shape, np.array(p['output']).shape))
    return len(s) == 1, list(s)[0] if len(s)==1 else None

def detect_identity(pairs):
    ok, sz = _all_same_size(pairs)
    if not ok: return False
    return all(np.array_equal(np.array(p['input']), np.array(p['output'])) for p in pairs)

def detect_color_remap(pairs):
    ok, sz = _all_same_size(pairs)
    if not ok or sz[0] != sz[1]: return None
    cmap = {}
    for p in pairs:
        a, b = np.array(p['input']).flatten(), np.array(p['output']).flatten()
        for s, d in zip(a, b):
            s, d = int(s), int(d)
            if s in cmap and cmap[s] != d: return None
            cmap[s] = d
    return cmap if cmap else None

def detect_rotation(pairs):
    for angle, k in [(90,1),(180,2),(270,3)]:
        if all(np.array_equal(np.rot90(np.array(p['input']),k), np.array(p['output']))
               for p in pairs):
            return angle
    return None

def detect_flip(pairs):
    for ax, name in [(1,'hflip'),(0,'vflip'),(-1,'hvflip')]:
        def flipped(ig, ax=ax):
            if ax == -1: return np.flip(np.flip(ig,0),1)
            return np.flip(ig, ax)
        if all(np.array_equal(flipped(np.array(p['input'])), np.array(p['output']))
               for p in pairs):
            return name
    return None

def detect_transpose(pairs):
    return all(np.array_equal(np.array(p['input']).T, np.array(p['output'])) for p in pairs)

def detect_antitranspose(pairs):
    return all(np.array_equal(np.fliplr(np.flipud(np.array(p['input']))).T,
                              np.array(p['output'])) for p in pairs)

def detect_scale(pairs):
    for f in [2,3,4,5]:
        if all(np.array_equal(
                np.repeat(np.repeat(np.array(p['input']),f,0),f,1),
                np.array(p['output'])) for p in pairs):
            return f
    return None

def detect_tile(pairs):
    p0 = pairs[0]
    ih,iw = np.array(p0['input']).shape
    oh_,ow_ = np.array(p0['output']).shape
    if oh_%ih!=0 or ow_%iw!=0: return None
    tr,tc = oh_//ih, ow_//iw
    if all(np.array_equal(np.tile(np.array(p['input']),(tr,tc)),
                          np.array(p['output'])) for p in pairs):
        return tr, tc
    return None

def detect_crop(pairs):
    """Fixed crop: output = input[r0:r0+oh, c0:c0+ow] for all pairs."""
    p0 = pairs[0]
    ig0, og0 = np.array(p0['input']), np.array(p0['output'])
    ih, iw = ig0.shape; oh_,ow_ = og0.shape
    if oh_>ih or ow_>iw: return None
    for r0 in range(ih-oh_+1):
        for c0 in range(iw-ow_+1):
            if np.array_equal(ig0[r0:r0+oh_, c0:c0+ow_], og0):
                if all(np.array_equal(
                        np.array(p['input'])[r0:r0+oh_,c0:c0+ow_],
                        np.array(p['output'])) for p in pairs[1:]):
                    return r0, c0, oh_, ow_
    return None

def detect_const(pairs):
    outs = [str(p['output']) for p in pairs]
    if len(set(outs)) == 1:
        return g2t(pairs[0]['output'])[np.newaxis]  # (1,C,H,W)
    return None

def detect_pixel_permutation(pairs):
    """Detect a consistent pixel shuffle (works for square and rectangular equal-size grids)."""
    ok, sz = _all_same_size(pairs)
    if not ok or sz[0] != sz[1]: return None
    ih, iw = sz[0]
    if ih != H or iw != W: return None  # only works for 30×30 currently
    # build candidate from first pair using unique values
    src = np.array(pairs[0]['input']).flatten()
    dst = np.array(pairs[0]['output']).flatten()
    mapping = {}
    for di, dv in enumerate(dst):
        cands = np.where(src == dv)[0]
        if len(cands) != 1: return None
        mapping[di] = cands[0]
    idx = np.array([mapping[i] for i in range(H*W)], np.int64)
    for p in pairs[1:]:
        s = np.array(p['input']).flatten()
        if not np.array_equal(s[idx], np.array(p['output']).flatten()): return None
    return idx

print('Pattern detectors ready.')

In [ ]:
# ── Linear (least-squares) solver for small grids ──────────────────

def model_linear_onnx(W_np, in_dim, out_dim):
    """Build Reshape→Gemm→Reshape ONNX model."""
    sh_flat = onh.from_array(np.array([1, in_dim], np.int64), name='sh_flat')
    sh_out  = onh.from_array(np.array([1, C, H, W], np.int64), name='sh_out')
    wt      = onh.from_array(W_np.astype(np.float32), name='W')
    nodes = [
        oh.make_node('Constant',[], ['sh_flat'], value=sh_flat),
        oh.make_node('Reshape', ['input','sh_flat'], ['inp_flat']),
        oh.make_node('Constant',[], ['W'], value=wt),
        oh.make_node('Gemm', ['inp_flat','W'], ['out_flat'],
                     transB=0, alpha=1.0, beta=0.0),
        oh.make_node('Constant',[], ['sh_out'], value=sh_out),
        oh.make_node('Reshape', ['out_flat','sh_out'], ['output']),
    ]
    return make_model(nodes)

def try_linear_solve(all_pairs):
    """Least-squares on all data. Returns ONNX model or None."""
    ok, sz = _all_same_size(all_pairs)
    if not ok: return None
    (ih,iw),(oh_,ow_) = sz
    in_dim  = C*ih*iw   # use actual grid size (padded in tensor)
    out_dim = C*oh_*ow_
    # File-size check: W matrix = in_dim * out_dim * 4 bytes
    if in_dim * out_dim * 4 > MAX_BYTES: return None
    n = len(all_pairs)
    if n < 2: return None
    
    # Build X (n, in_dim) and Y (n, out_dim) from the real grid sizes
    def g2small(grid, h, w):
        t = np.zeros((C, h, w), np.float32)
        for r in range(h):
            for cc in range(w):
                t[grid[r][cc], r, cc] = 1.0
        return t.flatten()
    
    X = np.stack([g2small(p['input'],  ih, iw) for p in all_pairs])
    Y = np.stack([g2small(p['output'], oh_, ow_) for p in all_pairs])
    
    try:
        W, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
        err = np.max(np.abs((X@W).round() - Y))
        if err > 0.4: return None
    except Exception:
        return None
    
    # W shape: (in_dim, out_dim). Gemm needs (1,in_dim) @ (in_dim, out_dim) → (1, out_dim)
    # We need to embed in full C,H,W space for the ONNX I/O contract.
    # Build a padded W: (C*H*W) → (C*H*W) that operates on the top-left subgrid.
    W_full = np.zeros((C*H*W, C*H*W), np.float32)
    # Map from small input coords to full tensor coords
    def idx_full(ch, r, c, w_full=W):  return ch*H*W + r*W + c
    def idx_small_in(ch, r, c):        return ch*iw*ih + r*iw + c   # wait, let me do this properly
    
    # Small in: (C, ih, iw) flattened row-major → index ch*ih*iw + r*iw + c
    # Full in:  (C, H, W)  flattened            → index ch*H*W  + r*W  + c
    # Build selection matrices
    # P_in  : (in_dim, C*H*W) — selects relevant input pixels
    # P_out : (C*H*W, out_dim) — places output pixels back
    P_in = np.zeros((in_dim, C*H*W), np.float32)
    for ch in range(C):
        for r in range(ih):
            for c in range(iw):
                small_i = ch*ih*iw + r*iw + c
                full_i  = ch*H*W   + r*W  + c
                P_in[small_i, full_i] = 1.0
    
    P_out = np.zeros((C*H*W, out_dim), np.float32)
    for ch in range(C):
        for r in range(oh_):
            for c in range(ow_):
                full_i  = ch*H*W    + r*W   + c
                small_i = ch*oh_*ow_ + r*ow_ + c
                P_out[full_i, small_i] = 1.0
    
    W_full_mat = P_in.T @ W @ P_out.T  # (C*H*W, C*H*W)
    if W_full_mat.nbytes > MAX_BYTES: return None
    return model_linear_onnx(W_full_mat.T, C*H*W, C*H*W)

print('Linear solver ready.')

In [ ]:
# ── Trained convolution solver ─────────────────────────────────────
import inspect as _inspect

def _onnx_export(model, path):
    model.cpu().eval()
    dummy = torch.randn(1, C, H, W)
    kw = dict(opset_version=17, input_names=['input'], output_names=['output'])
    if 'dynamo' in _inspect.signature(torch.onnx.export).parameters:
        kw['dynamo'] = False
    torch.onnx.export(model, dummy, str(path), **kw)


class ConvNet(nn.Module):
    """Flexible conv network: tiers 1-5."""
    def __init__(self, kind, hidden=8, k=3, blocks=2):
        super().__init__()
        p = k//2
        if kind == 'conv1':   # single layer
            self.net = nn.Conv2d(C, C, k, padding=p, bias=False)
        elif kind == 'bneck': # bottleneck
            self.net = nn.Sequential(
                nn.Conv2d(C,hidden,1,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,hidden,k,padding=p,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,C,1,bias=False))
        elif kind == 'res':   # residual
            layers = [nn.Conv2d(C,hidden,k,padding=p,bias=False), nn.ReLU()]
            for _ in range(blocks-1):
                layers += [nn.Conv2d(hidden,hidden,k,padding=p,bias=False), nn.ReLU()]
            layers.append(nn.Conv2d(hidden,C,k,padding=p,bias=False))
            self.body = nn.Sequential(*layers)
            self.net  = None
        elif kind == 'dilated':
            self.net = nn.Sequential(
                nn.Conv2d(C,hidden,k,padding=p,dilation=1,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,hidden,k,padding=p*2,dilation=2,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,hidden,k,padding=p*4,dilation=4,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,C,1,bias=False))
        elif kind == 'global':
            self.local  = nn.Sequential(
                nn.Conv2d(C,hidden,3,padding=1,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,hidden,3,padding=1,bias=False), nn.ReLU())
            self.gpool  = nn.AdaptiveAvgPool2d(1)
            self.gfc    = nn.Sequential(nn.Conv2d(C,hidden,1,bias=False), nn.ReLU())
            self.fuse   = nn.Conv2d(hidden*2,C,1,bias=False)
            self.net    = None
        self.kind = kind

    def forward(self, x):
        if self.kind == 'res':
            return x + self.body(x)
        elif self.kind == 'global':
            lf = self.local(x)
            gf = self.gfc(self.gpool(x)).expand_as(lf)
            return self.fuse(torch.cat([lf,gf],1))
        return self.net(x)


ARCH_LIST = [
    # (name, params_to_construct)
    ('conv1',   dict(kind='conv1', k=1)),
    ('conv3',   dict(kind='conv1', k=3)),
    ('conv5',   dict(kind='conv1', k=5)),
    ('conv7',   dict(kind='conv1', k=7)),
    ('bnk4k3',  dict(kind='bneck', hidden=4,  k=3)),
    ('bnk4k5',  dict(kind='bneck', hidden=4,  k=5)),
    ('bnk8k3',  dict(kind='bneck', hidden=8,  k=3)),
    ('bnk8k5',  dict(kind='bneck', hidden=8,  k=5)),
    ('bnk16k3', dict(kind='bneck', hidden=16, k=3)),
    ('res8b2',  dict(kind='res',   hidden=8,  k=3, blocks=2)),
    ('res16b2', dict(kind='res',   hidden=16, k=3, blocks=2)),
    ('res8b3',  dict(kind='res',   hidden=8,  k=3, blocks=3)),
    ('dil8',    dict(kind='dilated',hidden=8, k=3)),
    ('dil16',   dict(kind='dilated',hidden=16,k=3)),
    ('glob8',   dict(kind='global', hidden=8)),
    ('glob16',  dict(kind='global', hidden=16)),
]


def train_conv(all_pairs, arch_kwargs, max_epochs=2000, lr=0.003, patience=300,
               early_fail=200, batch=64):
    """Train a ConvNet. Returns (model_cpu, success)."""
    # Only same-size pairs
    for p in all_pairs:
        if np.array(p['input']).shape != np.array(p['output']).shape:
            return None, False
    
    X = torch.tensor(g2t_batch(all_pairs,'input')).to(DEVICE)
    Y = torch.tensor(g2t_batch(all_pairs,'output')).to(DEVICE)
    n = len(all_pairs)
    
    model = ConvNet(**arch_kwargs).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
    sch   = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=500, T_mult=2)
    
    best_correct, best_state, no_imp = 0, None, 0

    def accuracy():
        model.eval()
        with torch.no_grad():
            b = (model(X) > 0).float()
            return int(torch.all(b==Y, dim=(1,2,3)).sum().item())

    for epoch in range(max_epochs):
        model.train()
        if n <= batch:
            opt.zero_grad()
            loss = (nn.functional.binary_cross_entropy_with_logits(model(X),Y)
                    + 0.1*nn.functional.mse_loss(model(X), Y*2-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        else:
            perm = torch.randperm(n, device=DEVICE)
            for i in range(0, n, batch):
                idx = perm[i:i+batch]
                opt.zero_grad()
                loss = (nn.functional.binary_cross_entropy_with_logits(model(X[idx]),Y[idx])
                        + 0.1*nn.functional.mse_loss(model(X[idx]),Y[idx]*2-1))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        sch.step()

        if (epoch+1) % 25 == 0:
            c = accuracy()
            if (epoch+1) >= early_fail and best_correct == 0: return None, False
            if c > best_correct:
                best_correct = c
                best_state   = {k: v.cpu().clone() for k,v in model.state_dict().items()}
                no_imp = 0
            else:
                no_imp += 25
            if c == n: break
            if no_imp >= patience: break

    if best_state: model.load_state_dict(best_state)
    model.cpu()
    return model, (best_correct == n)

print('Trained conv solver ready.')

In [ ]:
# ── Master solver ──────────────────────────────────────────────────

def solve_task(task_data, task_num, verbose=False):
    """
    Returns (onnx_model_object, tier_name) or (None, None).
    Validates against ALL examples: train + test + arc-gen.
    """
    train = task_data.get('train', [])
    test  = task_data.get('test',  [])
    arcgen= task_data.get('arc-gen', [])
    all_p = train + test + arcgen    # full validation set
    tr_p  = train + test             # primary detection set

    def accept(model, tier):
        if check_all(model, all_p):
            if verbose: print(f'  {task_num:3d}: {tier}')
            return model, tier
        return None, None

    # 1 ─ identity
    if detect_identity(tr_p):
        r = accept(model_identity(), 'identity')
        if r[0]: return r

    # 2 ─ color remap
    cmap = detect_color_remap(tr_p)
    if cmap:
        r = accept(model_color_remap(cmap), 'color_remap')
        if r[0]: return r

    # 3 ─ geometric: rotation
    rot = detect_rotation(tr_p)
    if rot:
        mfn = {90:model_rot90, 180:model_rot180, 270:model_rot270}[rot]
        r = accept(mfn(), f'rot{rot}')
        if r[0]: return r

    # 4 ─ flip
    fl = detect_flip(tr_p)
    if fl:
        mfn = {'hflip':model_hflip,'vflip':model_vflip,'hvflip':model_hvflip}[fl]
        r = accept(mfn(), fl)
        if r[0]: return r

    # 5 ─ transpose / anti-transpose
    if detect_transpose(tr_p):
        r = accept(model_transpose(), 'transpose')
        if r[0]: return r
    if detect_antitranspose(tr_p):
        r = accept(model_antitranspose(), 'antitranspose')
        if r[0]: return r

    # 6 ─ scale-up
    sc = detect_scale(tr_p)
    if sc:
        r = accept(model_scale(sc), f'scale{sc}x')
        if r[0]: return r

    # 7 ─ tile
    tile = detect_tile(tr_p)
    if tile:
        r = accept(model_tile(*tile), f'tile{tile}')
        if r[0]: return r

    # 8 ─ crop
    crop = detect_crop(tr_p)
    if crop:
        r = accept(model_crop(*crop), f'crop{crop[:2]}')
        if r[0]: return r

    # 9 ─ pixel permutation
    perm = detect_pixel_permutation(tr_p)
    if perm is not None:
        r = accept(model_gather_custom(perm), 'pixel_perm')
        if r[0]: return r

    # 10 ─ constant output
    ct = detect_const(tr_p)
    if ct is not None:
        r = accept(model_const(ct), 'const_output')
        if r[0]: return r

    # 11 ─ color_remap + rotation combos
    for angle in [90, 180, 270]:
        rotfn = {90: lambda g: np.rot90(g,1), 180: lambda g: np.rot90(g,2),
                  270: lambda g: np.rot90(g,3)}[angle]
        rotmod= {90:model_rot90, 180:model_rot180, 270:model_rot270}[angle]
        faux  = [{'input': p['input'], 'output': rotfn(np.array(p['output'])).tolist()}
                  for p in tr_p if np.array(p['input']).shape == np.array(p['output']).shape]
        if faux and len(faux)==len(tr_p):
            cm2 = detect_color_remap(faux)
            if cm2:
                # model: color_remap then rotate
                # We need a combined ONNX: remap first, then rotate-gather
                # Build: input → conv(remap) → gather(rotate) → output
                W2 = np.zeros((C,C,1,1),np.float32)
                for s,d in cm2.items(): W2[d,s,0,0]=1.0
                for ch in range(C):
                    if ch not in cm2: W2[ch,ch,0,0]=1.0
                k2 = angle//90
                idx2 = _perm_from_fn({90:lambda r,c:(c,H-1-r),
                                       180:lambda r,c:(H-1-r,W-1-c),
                                       270:lambda r,c:(W-1-c,r)}[angle])
                sh_c = onh.from_array(np.array([1,C,H*W],np.int64),name='sh_c')
                sh_o = onh.from_array(np.array([1,C,H,W],np.int64),name='sh_o')
                wt2  = onh.from_array(W2, name='W2')
                gi2  = onh.from_array(idx2, name='gi2')
                combo_nodes = [
                    oh.make_node('Constant',[],['W2'],value=wt2),
                    oh.make_node('Conv',['input','W2'],['remapped'],kernel_shape=[1,1],pads=[0,0,0,0]),
                    oh.make_node('Constant',[],['sh_c'],value=sh_c),
                    oh.make_node('Reshape',['remapped','sh_c'],['flat2']),
                    oh.make_node('Constant',[],['gi2'],value=gi2),
                    oh.make_node('Gather',['flat2','gi2'],['gath2'],axis=2),
                    oh.make_node('Constant',[],['sh_o'],value=sh_o),
                    oh.make_node('Reshape',['gath2','sh_o'],['output']),
                ]
                combo_model = make_model(combo_nodes)
                r = accept(combo_model, f'remap+rot{angle}')
                if r[0]: return r

    # 12 ─ linear least-squares (small grids only)
    lin = try_linear_solve(all_p)
    if lin:
        r = accept(lin, 'linear')
        if r[0]: return r

    # 13 ─ trained single-layer conv (1x1, 3x3, 5x5, 7x7) — ALL data
    for name, arch_kw in ARCH_LIST:
        model_t, success = train_conv(all_p, arch_kw)
        if not success: continue
        tmp_path = SUB_DIR / f'_tmp_{task_num}.onnx'
        _onnx_export(model_t, tmp_path)
        if tmp_path.stat().st_size > MAX_BYTES:
            tmp_path.unlink()
            continue
        passed, total = run_onnx(tmp_path, all_p)
        if passed == total:
            if verbose: print(f'  {task_num:3d}: trained({name})')
            m = onnx.load(str(tmp_path))
            tmp_path.unlink()
            return m, f'trained_{name}'
        tmp_path.unlink()

    if verbose: print(f'  {task_num:3d}: UNSOLVED')
    return None, None

print('Master solver ready.')

In [ ]:
# ── Run all 400 tasks ──────────────────────────────────────────────

task_files = sorted(TASK_DIR.glob('task*.json'))
print(f'Found {len(task_files)} tasks')

results  = []
solved   = 0
tot_score= 0.0
t_start  = time.time()
log_rows = []

for tf in task_files:
    task_num = int(tf.stem.replace('task',''))
    t0 = time.time()

    try:
        task_data = load_task(tf)
        model, tier = solve_task(task_data, task_num, verbose=False)
    except Exception as e:
        model, tier = None, None

    dt = time.time() - t0

    if model is not None:
        out_path = SUB_DIR / f'task{task_num:03d}.onnx'
        try:
            save_onnx(model, out_path)
            sc = score_model(out_path)
        except Exception:
            sc = 1.0
        solved    += 1
        tot_score += sc
        tag = 'OK'
        log_rows.append((task_num, tier, f'{sc:.2f}', f'{dt:.1f}s'))
    else:
        tag = '--'
        log_rows.append((task_num, 'FAIL', '0', f'{dt:.1f}s'))

    results.append({'task': task_num, 'tier': tier, 'solved': model is not None,
                    'score': round(tot_score,2)})

    # progress every 10 tasks
    if task_num % 10 == 0:
        elapsed = time.time()-t_start
        eta = elapsed/task_num*(400-task_num) if task_num > 0 else 0
        print(f'Task {task_num:3d} {tag:2s} | solved {solved}/{task_num} '
              f'| score {tot_score:.1f} | {elapsed/60:.1f}m elapsed | ETA {eta/60:.1f}m')

elapsed = time.time()-t_start
print(f'\n{"="*60}')
print(f'Solved:    {solved}/400')
print(f'Score:     {tot_score:.1f}')
print(f'Time:      {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'{"="*60}')

In [ ]:
# ── Tier breakdown ─────────────────────────────────────────────────
from collections import Counter
tier_counts = Counter(r['tier'] for r in results if r['tier'])
print('Solutions by tier:')
for t, c in tier_counts.most_common(20):
    print(f'  {t:25s}: {c}')

In [ ]:
# ── Create submission.zip ──────────────────────────────────────────
onnx_files = sorted(SUB_DIR.glob('task*.onnx'))
zip_path   = OUTPUT_DIR / 'submission.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in onnx_files:
        zf.write(f, f.name)

print(f'submission.zip: {len(onnx_files)} files, '
      f'{zip_path.stat().st_size/1024:.0f} KB')

# Save log
with open(OUTPUT_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Done — submit /kaggle/working/submission.zip')